# VINE: four inpainting methods

Repository: [https://github.com/Shilin-LU/VINE](https://github.com/Shilin-LU/VINE)

Paper: [https://arxiv.org/abs/2410.18775](https://arxiv.org/abs/2410.18775)

Inspected commit: `d72f11a93f8de9bc45ef61202b9c8dda2a70cc77`

Watermark stage: `post_hoc`
Domain: image (framewise video adapter)

This method can be applied to existing media, so the notebook uses the fixed MOSE sample.

ProPainter, DiffuEraser, EffectErase, and E2FGVI-HQ receive the same authentic watermarked frames and binary mask. One official DiffuEraser execution is used: its internal ProPainter prior is preserved as the ProPainter result, and its final refinement is the DiffuEraser result. Inputs shorter than 22 frames are padded by repeating only the last frame/mask to 24 for that pipeline, then both reported outputs are trimmed to the authentic length. EffectErase applies its released Wan 4n+1 temporal adapter; E2FGVI-HQ uses the authors' official arbitrary-resolution test entry point.

VINE's official list encoder center-crops non-square images. The adapter applies that exact crop to every MOSE frame and mask, embeds one fixed 100-bit text payload, and records the geometry. The released decoder has no pixel localizer, so only bit recovery is reported.


## Authenticity contract

- Watermark preparation: Extract the video frames, run the official VINE-R list encoder with one fixed message, and reassemble the encoded frames at the source frame rate.
- Checkpoint status: READY: VINE-R and VINE-B encoder/decoder checkpoints are published on Hugging Face.
- Protocol note: VINE center-crops non-square inputs in its demo.  The experiment must record the spatial adapter and align the object-removal mask to the watermarked frame geometry.
- No result cell fabricates metrics. Native extraction must finish and write its JSON before values are displayed.
- A method without a released spatial detector reports bit recovery only; it is not assigned a proxy localizer.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/remaining_watermark_methods.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"

def run(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", shlex.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def require(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path

def require_commit(repository, expected):
    actual = subprocess.check_output(["git", "-C", str(repository), "rev-parse", "HEAD"], text=True).strip()
    if actual != expected:
        raise RuntimeError(f"Expected {expected} in {repository}, found {actual}")

def show_results(run_root):
    run_root = Path(run_root)
    for name in ("watermarked.mp4", "aligned-mask.mp4"):
        path = run_root / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    attack = run_root / "attack"
    for name in (
        "input-watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4",
        "effecterase.mp4", "e2fgvi.mp4",
    ):
        path = attack / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    result_json = {
        *run_root.rglob("summary.json"),
        *(attack / name for name in ("sample.json", "removal-pair.json", "effecterase.json", "e2fgvi.json")),
    }
    for path in sorted(path for path in result_json if path.is_file()):
        print(path.relative_to(run_root))
        display(JSON(json.loads(path.read_text())))

print("Repository:", ROOT)


## Canonical one-clip comparison: all four inpainters

This is the canonical comparison for the minimum one-clip requirement. All four removers use the same frozen watermarked input and aligned binary mask. Every remover is presented in the same order, with the same output-level fields and the same video/JSON evidence. Method-specific resizing, padding, and mask dilation remain recorded in JSON rather than being silently normalized.

| Inpainting method | Status | Runtime (s) | Frames | Dimensions | FPS | Bytes | Evidence |
|---|---:|---:|---:|---:|---:|---:|---|
| ProPainter | complete | not recorded | 67 | 480 × 480 | 25.000 | 437,480 | [video](../../runs/remaining-watermarks/vine/02b72b8d/attack/propainter.mp4) · [JSON](../../runs/remaining-watermarks/vine/02b72b8d/attack/propainter.json) |
| DiffuEraser | complete | not recorded | 67 | 480 × 480 | 25.000 | 532,990 | [video](../../runs/remaining-watermarks/vine/02b72b8d/attack/diffueraser.mp4) · [JSON](../../runs/remaining-watermarks/vine/02b72b8d/attack/diffueraser.json) |
| EffectErase | complete | 115.154 | 67 | 480 × 480 | 25.000 | 252,569 | [video](../../runs/remaining-watermarks/vine/02b72b8d/attack/effecterase.mp4) · [JSON](../../runs/remaining-watermarks/vine/02b72b8d/attack/effecterase.json) |
| E2FGVI-HQ | complete | 14.032 | 67 | 480 × 480 | 25.000 | 644,161 | [video](../../runs/remaining-watermarks/vine/02b72b8d/attack/e2fgvi.mp4) · [JSON](../../runs/remaining-watermarks/vine/02b72b8d/attack/e2fgvi.json) |

Watermark method: **Vine**. Runtime is shown only when the original run record captured it; `not recorded` is not replaced with an estimate. Watermark-specific localization or recovery metrics are downstream evaluations and are kept separate from this equal inpainting comparison.


In [ ]:
from pathlib import Path
import json
from IPython.display import JSON, Markdown, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/finalize_four_inpainter_notebooks.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
ATTACK = ROOT / 'runs/remaining-watermarks/vine/02b72b8d/attack'
for name in ("input-watermarked.mp4", "watermarked.mp4", "gt-mask.mp4"):
    path = ATTACK / name
    if path.is_file():
        print(name)
        display(Video(str(path), embed=True, html_attributes="controls muted loop"))

REMOVERS = (
    ("ProPainter", "propainter"),
    ("DiffuEraser", "diffueraser"),
    ("EffectErase", "effecterase"),
    ("E2FGVI-HQ", "e2fgvi"),
)
for label, stem in REMOVERS:
    display(Markdown(f"### {label}"))
    video = ATTACK / f"{stem}.mp4"
    metadata = ATTACK / f"{stem}.json"
    if not video.is_file() or not metadata.is_file():
        raise FileNotFoundError(f"Missing {label} evidence in {ATTACK}")
    display(Video(str(video), embed=True, html_attributes="controls muted loop"))
    display(JSON(json.loads(metadata.read_text())))

for name in ("sample.json", "removal-pair.json"):
    metadata = ATTACK / name
    if metadata.is_file():
        print(name)
        display(JSON(json.loads(metadata.read_text())))


In [ ]:
VINE = Path("/path/to/VINE")
VINE_PYTHON = VINE / ".venv/bin/python"
DIFFUERASER = Path("/path/to/DiffuEraser")
DIFFUERASER_PYTHON = DIFFUERASER / ".venv/bin/python"
GPU = "0"
MOSE_VIDEO = ROOT / "evidence/mose-task1/review50/media/02b72b8d/original.mp4"
MOSE_MASK = ROOT / "evidence/mose-task1/review50/media/02b72b8d/gt_mask.mp4"
RUN_ROOT = ROOT / "runs/remaining-watermarks/vine/02b72b8d"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WATERMARKED_VIDEO = RUN_ROOT / "watermarked.mp4"
ALIGNED_MASK = RUN_ROOT / "aligned-mask.mp4"


In [ ]:
require_commit(VINE, "d72f11a93f8de9bc45ef61202b9c8dda2a70cc77")
require(VINE_PYTHON, "VINE Python")


In [ ]:
run([
    VINE_PYTHON, SCRIPTS / "vine_video.py", "embed", "--source", VINE,
    "--input-video", MOSE_VIDEO, "--input-mask", MOSE_MASK,
    "--output-video", WATERMARKED_VIDEO, "--output-mask", ALIGNED_MASK,
    "--encoder-model", "Shilin-LU/VINE-R-Enc", "--message", "Hello World!",
])


## Run all four inpainting methods on the frozen input

The preparation cell freezes hashes, frame count, dimensions, FPS, mask foreground counts, provenance, and any short-video padding. Every remover below consumes that frozen input and mask.


In [ ]:
ATTACK_RUN = RUN_ROOT / "attack"
if not (ATTACK_RUN / "sample.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "prepare_watermark_removal_sample.py", "prepare",
        "--source-video", WATERMARKED_VIDEO, "--source-mask", ALIGNED_MASK,
        "--output", ATTACK_RUN, "--watermark-method", "VINE",
        "--sample-id", "02b72b8d", "--sample-source", "MOSE fixed sample after upstream center crop",
        "--source-kind", "mose_post_hoc",
    ])
else:
    print("Prepared sample already exists:", ATTACK_RUN / "sample.json")


In [ ]:
if not (ATTACK_RUN / "removal-pair.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "run_watermark_removal_pair.py",
        "--run", ATTACK_RUN, "--diffueraser", DIFFUERASER,
        "--diffueraser-python", DIFFUERASER_PYTHON,
        "--device", GPU, "--seed", "2025",
    ])
else:
    print("Both removal outputs already exist:", ATTACK_RUN / "removal-pair.json")


In [ ]:
EFFECTERASE = Path(os.environ.get("EFFECTERASE", "/mnt/data_fast/smayan/research/EffectErase"))
EFFECTERASE_PYTHON = Path(os.environ.get("EFFECTERASE_PYTHON", str(EFFECTERASE / ".venv/bin/python")))
E2FGVI = Path(os.environ.get("E2FGVI", "/mnt/data_fast/smayan/research/E2FGVI"))
E2FGVI_CHECKPOINT = Path(os.environ.get(
    "E2FGVI_CHECKPOINT", str(E2FGVI / "release_model/E2FGVI-HQ-CVPR22.pth")
))
E2FGVI_PYTHON = Path(os.environ.get("E2FGVI_PYTHON", str(DIFFUERASER_PYTHON)))

if not (ATTACK_RUN / "effecterase.json").is_file():
    effect_env = os.environ.copy(); effect_env["CUDA_VISIBLE_DEVICES"] = GPU
    run([
        EFFECTERASE_PYTHON, SCRIPTS / "run_prepared_effecterase.py",
        "--source", EFFECTERASE, "--run", ATTACK_RUN, "--device", GPU,
    ], env=effect_env)
else:
    print("EffectErase output already exists:", ATTACK_RUN / "effecterase.json")

if not (ATTACK_RUN / "e2fgvi.json").is_file():
    run([
        E2FGVI_PYTHON, SCRIPTS / "run_prepared_e2fgvi.py",
        "--source", E2FGVI, "--checkpoint", E2FGVI_CHECKPOINT,
        "--run", ATTACK_RUN, "--python", E2FGVI_PYTHON, "--device", GPU,
    ])
else:
    print("E2FGVI-HQ output already exists:", ATTACK_RUN / "e2fgvi.json")


In [ ]:
run([
    VINE_PYTHON, SCRIPTS / "vine_video.py", "detect", "--source", VINE,
    "--decoder-model", "Shilin-LU/VINE-R-Dec", "--message", "Hello World!",
    "--output", RUN_ROOT / "detection",
    "--case", f"original={ATTACK_RUN / 'input-watermarked.mp4'}",
    "--case", f"propainter={ATTACK_RUN / 'propainter.mp4'}",
    "--case", f"diffueraser={ATTACK_RUN / 'diffueraser.mp4'}",
    "--case", f"effecterase={ATTACK_RUN / 'effecterase.mp4'}",
    "--case", f"e2fgvi={ATTACK_RUN / 'e2fgvi.mp4'}",
])


In [ ]:
show_results(RUN_ROOT)
